# Klebsiella metadata

In [1]:
import pandas as pd
#paths

# No additional imports are needed; you already imported 'pandas as pd' above.
data_dir = "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david"
metadata_file = data_dir + "/final/metadata_final_curated_slimmed.tsv"

metadata = pd.read_csv(metadata_file, sep="\t", low_memory=False)
display(metadata.head())

,Sample,is_kpsc,kpsc_final_list,is_refseq,is_nctc,species,species_match,Clonal group,LINcode,Phylogroup,...,dev_stage,scientific_name,center_name,amr_study,study_setting,bakta_gbff_downloaded,bakta_gff3_downloaded,assembly_file,gff_file,poppunk_cluster
0,SAMD00002693,False,False,False,False,Klebsiella ornithinolytica,strong,NaN,NaN,NaN,...,NaN,Raoultella ornithinolytica,GIFU_MED,NaN,NaN,False,True,NaN,NaN,NaN
1,SAMD00008836,False,False,False,False,Klebsiella terrigena,strong,NaN,NaN,NaN,...,NaN,Klebsiella pneumoniae subsp. ozaenae,GIFU_MED,NaN,NaN,False,True,NaN,david/raw/klebsiella_gff3/SAMD00008836.bakta.g...,NaN
2,SAMD00008944,False,False,False,False,Klebsiella oxytoca,strong,NaN,NaN,NaN,...,NaN,Klebsiella oxytoca,GIFU_MED,NaN,NaN,False,True,NaN,david/raw/klebsiella_gff3/SAMD00008944.bakta.g...,NaN
3,SAMD00009480,False,False,False,False,Klebsiella aerogenes,strong,NaN,NaN,NaN,...,NaN,Klebsiella aerogenes,GIFU_MED,NaN,NaN,False,True,NaN,david/raw/klebsiella_gff3/SAMD00009480.bakta.g...,NaN
4,SAMD00049595,True,True,False,False,Klebsiella pneumoniae,strong,CG711,0_0_137_11_45_0_0_0_1_0,Kp1,...,NaN,Klebsiella pneumoniae,KYOTO_GM,NaN,NaN,False,True,seb/assemblies_2/klebsiella_pneumoniae__01/SAM...,david/raw/klebsiella_gff3/SAMD00049595.bakta.g...,128


In [2]:
# Check how many gff files are missing
print(f"Number of gff files missing: {metadata['gff_file'].isna().sum()}")
missing_gff_files = metadata[metadata['gff_file'].isna()]
display(missing_gff_files.head())

Number of gff files missing: 1


,Sample,is_kpsc,kpsc_final_list,is_refseq,is_nctc,species,species_match,Clonal group,LINcode,Phylogroup,...,dev_stage,scientific_name,center_name,amr_study,study_setting,bakta_gbff_downloaded,bakta_gff3_downloaded,assembly_file,gff_file,poppunk_cluster
0,SAMD00002693,False,False,False,False,Klebsiella ornithinolytica,strong,NaN,NaN,NaN,...,NaN,Raoultella ornithinolytica,GIFU_MED,NaN,NaN,False,True,NaN,NaN,NaN


In [3]:
# Double check that all with sublineage not null are in kpsc_final_list
sl_samples = metadata[metadata['Sublineage'].notna()]
# Number of samples with sublineage not null
print(f"Number of samples with sublineage not null: {len(sl_samples)}")
# Number of samples with sublineage not null and in kpsc_final_list
sl_samples = sl_samples[sl_samples['kpsc_final_list']]
print(f"Number of samples with sublineage not null and in kpsc_final_list: {len(sl_samples)}")

Number of samples with sublineage not null: 80641
Number of samples with sublineage not null and in kpsc_final_list: 79042


In [4]:
# Check the assembly column completeness - 'assembly_file'
print(f"Number of metadata entries: {len(metadata)}")
print(f"Number of missing assembly files: {metadata['assembly_file'].isna().sum()}")
# Group by species and count, sort by count
missing_assembly_files = metadata[metadata['assembly_file'].isna()]
print(missing_assembly_files['species'].value_counts().head(10))

Number of metadata entries: 88303
Number of missing assembly files: 3148
species
Klebsiella michiganensis      846
Klebsiella aerogenes          684
Klebsiella oxytoca            548
Klebsiella ornithinolytica    458
Klebsiella planticola         215
Klebsiella grimontii          180
Klebsiella terrigena           60
Klebsiella pasteurii           54
Klebsiella Ka4                 35
Klebsiella sauropsida          19
Name: count, dtype: int64


In [5]:
# Filter to those samples in kpsc_final_list True
kpsc_final_list_samples = metadata[metadata['kpsc_final_list']]
print(f"Number of samples in kpsc_final_list: {len(kpsc_final_list_samples)}")

Number of samples in kpsc_final_list: 79450


In [6]:
# The size of the SLs, value counts
# Calculate the total number of samples in sublineages of size < 25
sublineage_counts = kpsc_final_list_samples['Sublineage'].value_counts()
# Print a list of all sl's with size > 100,convert to list so we can print it
print(list(sublineage_counts[sublineage_counts > 500].index))
display(sublineage_counts.head(23))

['SL258', 'SL147', 'SL17', 'SL307', 'SL15', 'SL14', 'SL37', 'SL45', 'SL101', 'SL231', 'SL268', 'SL29', 'SL39', 'SL23', 'SL395', 'SL35', 'SL107', 'SL13', 'SL111', 'SL25', 'SL48', 'SL34', 'SL3010']


Sublineage
SL258     16229
SL147      5096
SL17       4630
SL307      4449
SL15       3647
SL14       2521
SL37       1871
SL45       1803
SL101      1610
SL231      1049
SL268       954
SL29        909
SL39        886
SL23        863
SL395       851
SL35        783
SL107       702
SL13        647
SL111       601
SL25        590
SL48        567
SL34        531
SL3010      502
Name: count, dtype: int64

In [7]:
def run_panaroo_on_small_sublineages(n: int):
    # Calculate the total number of samples in sublineages of size < n
    small_sublineages = sublineage_counts[sublineage_counts < n]
    # Filter to metadata file to only include the SLs of size < n
    small_sublineages_metadata = kpsc_final_list_samples[kpsc_final_list_samples['Sublineage'].isin(small_sublineages.index)]
    print(f"Total number of samples in sublineages of size < {n}: {len(small_sublineages_metadata)}")
    print(f"Total number of sublineages of size < {n}: {len(small_sublineages)}")

    # Save the metadata file to david/processed/panaroo_run/sublineages_size_<n>.tsv
    small_sublineages_metadata.to_csv(f"/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/sublineages_smaller_than_{n}.tsv", sep="\t")
    print(f"Saved metadata file to david/processed/panaroo_run/sublineages_smaller_than_{n}.tsv")
    return small_sublineages_metadata

In [ ]:
# Define sublineage_other function

final_kpsc_samples = metadata[metadata['kpsc_final_list']]
print(f"Number of samples in final kpsc set: {len(final_kpsc_samples)}")

# It takes all metadata samples that are in final set, subtracts all samples in a list of Clonal groups provided
sublineage = "SL17"
# How many samples are in the sublineage
sublineage_samples = metadata[metadata['Sublineage'] == sublineage]
print(f"Number of samples in {sublineage}: {len(sublineage_samples)}")
# Subtract clonal groups from the sublineage
subtract_clonal_groups = ["CG15"]
sublineage_other_metadata = sublineage_samples[~sublineage_samples['Clonal group'].isin(subtract_clonal_groups)]
print(f"Number of samples in {sublineage} other: {len(sublineage_other_metadata)}")
# Save the metadata to david/processed/panaroo_run/{sublineage}_other.tsv
sublineage_other_metadata.to_csv(f"/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/{sublineage}_other.tsv", sep="\t")

Number of samples in final kpsc set: 79450
Number of samples in SL15: 3739
Number of samples in SL15 other: 59


In [15]:
# Pack sublineages (each < 500 samples) into greedy batches of >= 1500 total samples.
# We build batches from `final_kpsc_samples` only, and always save the final remainder batch.

out_dir = "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run"

target_total = 1000
max_sublineage_size = 500

batch_i = 0
current_sls = []
current_parts = []
current_count = 0

# Sublineage sample counts (within the final kpsc set)
sl_sample_counts = final_kpsc_samples["Sublineage"].value_counts().sort_values(ascending=False)

# Only keep SLs below the max size
sl_sample_counts_less_than_500 = sl_sample_counts[sl_sample_counts < max_sublineage_size]
sl_list = sl_sample_counts_less_than_500.index.tolist()

print(f"Found {len(sl_list)} sublineages with < {max_sublineage_size} samples.")

for sl in sl_list:
    sl_df = final_kpsc_samples.loc[final_kpsc_samples["Sublineage"] == sl]

    current_sls.append(sl)
    current_parts.append(sl_df)
    current_count += len(sl_df)

    # Save when we reach/exceed the target (including the SL that pushes it over)
    if current_count >= target_total:
        batch_df = pd.concat(current_parts, ignore_index=True)
        out_path = f"{out_dir}/next_sublineage_batch_{batch_i}.tsv"
        batch_df.to_csv(out_path, sep="\t")

        print(f"Saved batch {batch_i}: {current_count} samples across {len(current_sls)} sublineages -> {out_path}")

        batch_i += 1
        current_sls = []
        current_parts = []
        current_count = 0

# Save final remainder batch (often < 1500)
if current_parts:
    batch_df = pd.concat(current_parts, ignore_index=True)
    out_path = f"{out_dir}/next_sublineage_batch_{batch_i}.tsv"
    batch_df.to_csv(out_path, sep="\t")
    print(f"Saved final partial batch {batch_i}: {len(batch_df)} samples across {len(current_sls)} sublineages -> {out_path}")


Found 2864 sublineages with < 500 samples.
Saved batch 0: 1330 samples across 3 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/next_sublineage_batch_0.tsv
Saved batch 1: 1182 samples across 3 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/next_sublineage_batch_1.tsv
Saved batch 2: 1017 samples across 3 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/next_sublineage_batch_2.tsv
Saved batch 3: 1083 samples across 4 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/next_sublineage_batch_3.tsv
Saved batch 4: 1078 samples across 5 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/next_sublineage_batch_4.tsv
Saved batch 5: 1066 samples across 6 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/next_sublineage_batch_5.tsv
Saved batch 6: 1082 s